# Blast Radius — GLM-5.2 vs Claude Opus 4.8

**Long-context code comprehension benchmark.** We dump an entire repository into each model's context and ask, for a set of target symbols: *"if you rename this, every file and line that must change."* The answer is scored against a **deterministic oracle** built by static analysis (whole-token find-references), so the ground truth is independent of any model.

Two scoring levels:
- **File level** — did the model name the right *set of files*? (the question a refactoring engineer actually asks)
- **Line level** — did it pin the exact `(file, line)`? (stricter; the dump hands it line numbers, so this tests precise retrieval, not arithmetic)

This notebook only *visualizes* a results file. Generate one first from the terminal:
```bash
python run_benchmark.py            # real run (needs ANTHROPIC_API_KEY + GLM_API_KEY)
python run_benchmark.py --demo     # offline, synthesizes labeled fake answers
```

In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

RESULTS_DIR = Path("results")

# Load results/latest.json by default; point RESULTS_FILE elsewhere to compare runs.
RESULTS_FILE = RESULTS_DIR / "latest.json"
data = json.loads(RESULTS_FILE.read_text(encoding="utf-8"))

MODEL_COLORS = {"opus-4-8": "#C8643C", "glm-5-2": "#3C6EC8"}

# Only models that actually ran (skip those without an API key).
RAN = {k: m for k, m in data["models"].items() if "aggregate" in m}

banner = "  *** SYNTHETIC DEMO DATA — not real model output ***" if data["synthetic"] else ""
print(f"mode = {data['mode']}{banner}")
print(f"generated_at = {data['generated_at']}")
print(f"repo: {data['repo']['n_files']} files, {data['repo']['char_count']:,} chars "
      f"(~{data['repo']['est_tokens']:,} tokens)")
print(f"models that ran: {', '.join(m['label'] for m in RAN.values()) or '(none)'}")

## The oracle — ground truth blast radius per symbol

Sorted widest-first. This is the deterministic target both models are scored against.

In [ ]:
oracle_df = (
    pd.DataFrame(
        [{"symbol": s, "files": v["files"], "lines": v["lines"]}
         for s, v in data["oracle"].items()]
    )
    .sort_values("lines", ascending=False)
    .reset_index(drop=True)
)
oracle_df

## Headline: file- and line-level F1 (macro-averaged across symbols)

In [ ]:
rows = []
for key, m in RAN.items():
    agg = m["aggregate"]
    rows.append({
        "model": m["label"],
        "key": key,
        "file_F1": agg["file"]["macro"]["f1"],
        "file_precision": agg["file"]["macro"]["precision"],
        "file_recall": agg["file"]["macro"]["recall"],
        "line_F1": agg["line"]["macro"]["f1"],
        "line_precision": agg["line"]["macro"]["precision"],
        "line_recall": agg["line"]["macro"]["recall"],
        "mean_latency_s": agg["latency_s_mean"],
        "total_cost_usd": agg["total_cost_usd"],
    })
summary = pd.DataFrame(rows).set_index("model")
summary[["file_F1", "line_F1", "file_recall", "line_recall", "mean_latency_s", "total_cost_usd"]]

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4.2))
labels = list(summary.index)
x = range(len(labels))
w = 0.36
colors = [MODEL_COLORS.get(k, "#888") for k in summary["key"]]
ax.bar([i - w / 2 for i in x], summary["file_F1"], w, label="file-level F1", color=colors)
ax.bar([i + w / 2 for i in x], summary["line_F1"], w, label="line-level F1",
       color=colors, alpha=0.55, hatch="//")
ax.set_xticks(list(x)); ax.set_xticklabels(labels)
ax.set_ylim(0, 1.05); ax.set_ylabel("F1 (macro avg)")
ax.set_title("Blast-radius F1 by model")
for i, (f, l) in enumerate(zip(summary["file_F1"], summary["line_F1"])):
    ax.text(i - w / 2, f + 0.01, f"{f:.2f}", ha="center", va="bottom", fontsize=9)
    ax.text(i + w / 2, l + 0.01, f"{l:.2f}", ha="center", va="bottom", fontsize=9)
ax.legend(); fig.tight_layout(); plt.show()

## Precision vs recall (where do the models differ?)

Recall = did it find every reference? Precision = did it avoid inventing ones? A long-context model that 'reads' the whole repo should win on recall without paying in precision.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4.2), sharey=True)
for ax, level in zip(axes, ["file", "line"]):
    metrics = ["precision", "recall", "F1"]
    cols = [f"{level}_precision", f"{level}_recall", f"{level}_F1"]
    x = range(len(metrics)); w = 0.36
    for j, (model, row) in enumerate(summary.iterrows()):
        offs = (j - (len(summary) - 1) / 2) * w
        ax.bar([i + offs for i in x], [row[c] for c in cols], w, label=model,
               color=MODEL_COLORS.get(row["key"], "#888"))
    ax.set_xticks(list(x)); ax.set_xticklabels(metrics)
    ax.set_ylim(0, 1.05); ax.set_title(f"{level}-level")
axes[0].set_ylabel("score (macro avg)"); axes[0].legend()
fig.suptitle("Precision / Recall / F1"); fig.tight_layout(); plt.show()

## Does accuracy degrade as blast radius grows?

Per-symbol line-level recall, ordered by oracle size. The interesting failure mode for a long-context model is the wide-blast symbol (`UserProfile`, `AgentState`): can it still find *all 40+* references, or does recall fall off as the needle count rises?

In [ ]:
per_sym = []
for key, m in RAN.items():
    for s in m["per_symbol"]:
        per_sym.append({
            "model": m["label"], "key": key, "symbol": s["symbol"],
            "oracle_lines": s["oracle_lines"],
            "line_recall": s["line"]["recall"],
            "file_recall": s["file"]["recall"],
        })
per_sym_df = pd.DataFrame(per_sym)
order = oracle_df["symbol"].tolist()

fig, ax = plt.subplots(figsize=(11, 4.5))
for key, m in RAN.items():
    sub = per_sym_df[per_sym_df["key"] == key].set_index("symbol").reindex(order)
    ax.plot(order, sub["line_recall"], marker="o", label=m["label"],
            color=MODEL_COLORS.get(key, "#888"))
ax.set_ylim(0, 1.05); ax.set_ylabel("line-level recall")
ax.set_title("Per-symbol line recall (widest blast radius on the left)")
ax.set_xticklabels(order, rotation=40, ha="right")
ax.grid(axis="y", alpha=0.3); ax.legend(); fig.tight_layout(); plt.show()

## Cost and latency

The engineering trade-off behind the accuracy. Cost depends on each model's price (Opus 4.8 prompt-caches the shared dump across the 10 calls; set your real GLM price via `GLM_INPUT_PRICE` / `GLM_OUTPUT_PRICE`). Zero in demo mode.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
colors = [MODEL_COLORS.get(k, "#888") for k in summary["key"]]
axes[0].bar(summary.index, summary["mean_latency_s"], color=colors)
axes[0].set_title("Mean latency per query (s)"); axes[0].set_ylabel("seconds")
axes[1].bar(summary.index, summary["total_cost_usd"], color=colors)
axes[1].set_title("Total cost for the run (USD)"); axes[1].set_ylabel("USD")
for ax in axes:
    for i, v in enumerate(ax.containers[0].datavalues):
        ax.text(i, v, f"{v:.3g}", ha="center", va="bottom", fontsize=9)
fig.tight_layout(); plt.show()

if data["synthetic"]:
    print("NOTE: demo mode — latency and cost are zero (no API calls were made).")

## Read-out

Fill this in from your **live** run (the demo numbers are synthetic and exist only to prove the pipeline):

- **File-level F1** is the headline: can the model tell you which files to open for a refactor?
- **Line-level recall** is the long-context stress test: on a wide-blast symbol with 40+ references scattered across a 100k-token dump, does it find them *all*?
- **Precision** catches hallucinated references — the failure mode that makes an answer untrustworthy.
- **Cost/latency** is the lever that makes a long-context model like GLM-5.2 attractive *if* its accuracy holds.

The deterministic oracle means every number here is reproducible and auditable — re-run `run_benchmark.py` and the ground truth never moves.